# Trabajo Final Deep Learning — Módulo LLM / Chatbot RAG v7

Versión v7 alineada al aplicativo Angular + Node + Ollama.


## 0. Alcance del notebook

Este notebook reproduce el flujo del módulo LLM / Chatbot RAG usado en el aplicativo TF_DL_Grupo4.

La versión v7 incorpora los últimos ajustes del chatbot: selección de evidencia por intención, separación entre evidencia usada y reseñas recuperadas no usadas, reglas específicas por intención, prompt sin scores técnicos y respuestas enfocadas solo en la pregunta actual.


In [ ]:
# ============================================================
# Trabajo Final Deep Learning - Módulo LLM / Chatbot RAG
# Integrantes:
# - Romero Montoya, Renzo (E202510085)
# - Fernandez Añazgo, Sissy (E201710373)
# - Zayerz Calvo, Jorge (E202510160)
# - Ramirez Lazaro, Alex (E202510516)
# - Condori Castillo, Edwin (E202510725)
#
# Dataset esperado: Excel con hojas Principal y Reviews.
# ============================================================

## 1. Instalación de librerías

In [ ]:
# En Colab, muchas librerías ya están instaladas.
# Se instalan solo las necesarias para lectura del Excel, limpieza y recuperación TF-IDF.

%pip -q install pandas openpyxl scikit-learn unidecode ipywidgets sentence-transformers numpy

# Proveedores opcionales de LLM:
# !pip -q install google-generativeai
# !pip -q install openai
# !pip -q install requests

Note: you may need to restart the kernel to use updated packages.


: 

## 2. Importación de librerías y configuración general

In [2]:
import os
import re
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from unidecode import unidecode
except Exception:
    def unidecode(x):
        return x

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

TOP_K = 5
MAX_CANDIDATES = 10
MAX_USED_REVIEWS = 5
RETRIEVER_MODE = "auto"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_REVIEWS_PER_LISTING = 120
MAX_CHARS_PER_CHUNK = 900
CHUNK_OVERLAP = 120

LLM_PROVIDER = "ollama"
GEMINI_MODEL = "gemini-1.5-flash"
OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL = "http://localhost:11434/api/generate"

EXCEL_SOURCE_PATHS = [Path("G4_mod_finale.xlsx"), Path(r"C:/TF_DL/G4_mod_finale.xlsx")]
APP_DATASET_JSON_PATHS = [Path("public/data/listings.json"), Path(r"C:/Users/andre/OneDrive/Documentos/Deep Learning/TF_DL_Grupo4/public/data/listings.json")]
APP_RAG_VERSION = "v7_app_alineado"
print("Versión RAG notebook:", APP_RAG_VERSION)


## 3. Carga del dataset

Sube el Excel actualizado cuando lo tengas.  
El notebook espera un archivo con una hoja parecida a **Principal** y otra parecida a **Reviews**.

In [28]:
def first_existing_path(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

DATA_PATH = first_existing_path(EXCEL_SOURCE_PATHS)
APP_DATASET_JSON = first_existing_path(APP_DATASET_JSON_PATHS)

if DATA_PATH is None:
    raise FileNotFoundError("No se encontró G4_mod_finale.xlsx.")

print("Excel fuente original usado:", DATA_PATH)
if APP_DATASET_JSON:
    print("JSON operativo de la app detectado:", APP_DATASET_JSON)
    with open(APP_DATASET_JSON, "r", encoding="utf-8") as f:
        app_dataset = json.load(f)
    print("JSON app:", app_dataset.get("meta", {}).get("listingCount"), "listings,", app_dataset.get("meta", {}).get("reviewCount"), "reseñas")
else:
    app_dataset = None
    print("No se detectó listings.json de la app; el notebook trabajará directamente desde Excel.")

sheets = pd.read_excel(DATA_PATH, sheet_name=None)
print("Hojas encontradas en Excel:")
for name, df in sheets.items():
    print(f"- {name}: {df.shape[0]} filas x {df.shape[1]} columnas")


Archivo usado: G4_mod_finale.xlsx
Hojas encontradas:
- Principal: 53 filas x 27 columnas
- Reviews: 2824 filas x 4 columnas


## 4. Funciones de limpieza y detección automática de columnas

In [4]:
def normalize_name(text):
    """Normaliza nombres de hojas/columnas para facilitar búsqueda robusta."""
    text = "" if text is None else str(text)
    text = unidecode(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_columns(df):
    """Limpia nombres de columnas sin perder el nombre original."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def make_unique_columns(columns):
    """Asegura nombres de columnas únicos y no vacíos."""
    seen = {}
    output = []

    for i, col in enumerate(columns):
        col = "" if pd.isna(col) else str(col).strip()
        if col == "" or col.lower() in ["nan", "none"]:
            col = f"col_{i}"

        base = col
        count = seen.get(base, 0)

        if count > 0:
            col = f"{base}_{count + 1}"

        seen[base] = count + 1
        output.append(col)

    return output


def promote_first_row_as_header_if_needed(df, expected_key="ID Airbnb"):
    """
    Corrige Excels donde la primera fila contiene los nombres reales de columnas.

    En el dataset del TF puede ocurrir que pandas lea columnas como:
    Unnamed: 0, Ítems:, Unnamed: 2, a, b, c...
    y que la fila 0 contenga:
    Ciudad, Distrito, Fecha de colecta, ID Airbnb, Alias...

    Esta función detecta ese patrón y promueve la fila 0 como encabezado.
    """
    df = df.copy()

    if df.empty:
        return df

    key = normalize_name(expected_key)
    normalized_cols = [normalize_name(c) for c in df.columns]

    key_in_columns = any(key == c or key in c for c in normalized_cols)

    first_row_values = df.iloc[0].tolist()
    first_row_norm = [normalize_name(x) for x in first_row_values]
    key_in_first_row = any(key == x or key in x for x in first_row_norm)

    placeholder_cols = sum(
        1 for c in normalized_cols
        if c.startswith("unnamed") or len(c) <= 2 or c in ["items"]
    )

    many_placeholder_cols = placeholder_cols >= max(2, len(df.columns) // 3)

    if key_in_first_row and (not key_in_columns or many_placeholder_cols):
        new_columns = make_unique_columns(first_row_values)
        df = df.iloc[1:].reset_index(drop=True)
        df.columns = new_columns
        print("Se corrigió encabezado: la primera fila fue promovida como nombre de columnas.")

    return df


def infer_sheet_name(sheets_dict, candidates):
    """Busca la hoja más probable según palabras candidatas."""
    normalized = {normalize_name(k): k for k in sheets_dict.keys()}

    # 1) Match exacto normalizado
    for cand in candidates:
        nc = normalize_name(cand)
        if nc in normalized:
            return normalized[nc]

    # 2) Match por contención
    for norm_sheet, original in normalized.items():
        for cand in candidates:
            nc = normalize_name(cand)
            if nc in norm_sheet or norm_sheet in nc:
                return original

    # 3) Fallback: primera hoja
    return list(sheets_dict.keys())[0]


def find_column(df, aliases, required=True):
    """
    Encuentra una columna por alias de forma robusta.
    Primero intenta match exacto normalizado, luego contención.
    """
    col_map = {normalize_name(c): c for c in df.columns}

    # Match exacto
    for alias in aliases:
        na = normalize_name(alias)
        if na in col_map:
            return col_map[na]

    # Match por contención
    for alias in aliases:
        na = normalize_name(alias)
        for norm_col, original_col in col_map.items():
            if na and (na in norm_col or norm_col in na):
                return original_col

    if required:
        raise ValueError(f"No se encontró columna para aliases: {aliases}\nColumnas disponibles: {list(df.columns)}")
    return None


def find_columns_by_keywords(df, keywords):
    """Devuelve columnas cuyo nombre contiene alguno de los keywords."""
    result = []
    for col in df.columns:
        ncol = normalize_name(col)
        if any(normalize_name(k) in ncol for k in keywords):
            result.append(col)
    return result


def safe_text(value):
    """Convierte valores a texto limpio."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text


# Inferencia de hojas
principal_sheet = infer_sheet_name(sheets, ["Principal", "Listado", "Listings", "Airbnb"])
reviews_sheet = infer_sheet_name(sheets, ["Reviews", "Reseñas", "Resenas", "Comentarios"])

principal_df = clean_columns(sheets[principal_sheet])
reviews_df = clean_columns(sheets[reviews_sheet])

# Corrección importante para el dataset del TF:
# si la primera fila contiene los nombres reales de columnas, se promueve como encabezado.
principal_df = promote_first_row_as_header_if_needed(principal_df, expected_key="ID Airbnb")
reviews_df = promote_first_row_as_header_if_needed(reviews_df, expected_key="ID Airbnb")

print("Hoja Principal detectada:", principal_sheet, principal_df.shape)
print("Hoja Reviews detectada:", reviews_sheet, reviews_df.shape)

# Inferencia de columnas ID
id_col_principal = find_column(
    principal_df,
    ["ID Airbnb", "id_airbnb", "listing_id", "id listing", "airbnb id", "id"],
    required=True
)

id_col_reviews = find_column(
    reviews_df,
    ["ID Airbnb", "id_airbnb", "listing_id", "id listing", "airbnb id", "id"],
    required=True
)

review_text_col = find_column(
    reviews_df,
    [
        "Reseña (Debe estar solo en español)",
        "Reseña",
        "Resena",
        "review",
        "comments",
        "comentario",
        "comentarios"
    ],
    required=True
)

print("Columna ID en Principal:", id_col_principal)
print("Columna ID en Reviews:", id_col_reviews)
print("Columna texto reseña:", review_text_col)
print("\nPrimeras columnas limpias de Principal:")
print(list(principal_df.columns[:10]))

Se corrigió encabezado: la primera fila fue promovida como nombre de columnas.
Hoja Principal detectada: Principal (52, 27)
Hoja Reviews detectada: Reviews (2824, 4)
Columna ID en Principal: ID Airbnb
Columna ID en Reviews: ID Airbnb
Columna texto reseña: Reseña (Debe estar solo en español)

Primeras columnas limpias de Principal:
['Ciudad', 'Distrito', 'Fecha de colecta', 'ID Airbnb', 'Alias', 'URL Canónica', 'Título de cabecera', 'Título de la propiedad', 'Resumen de la propiedad', 'Reconocimiento de Airbnb']


## 5. Exploración rápida del dataset

In [5]:
print("Vista rápida de Principal:")
display(principal_df.head(3))

print("\nVista rápida de Reviews:")
display(reviews_df.head(3))

principal_ids = set(principal_df[id_col_principal].dropna().astype(str))
review_ids = set(reviews_df[id_col_reviews].dropna().astype(str))
ids_with_reviews = principal_ids.intersection(review_ids)

print("\nResumen:")
print("Listados en Principal:", len(principal_ids))
print("Listados con reseñas asociadas:", len(ids_with_reviews))
print("Reseñas totales:", len(reviews_df))


def choose_demo_listing_id(prefer_with_reviews=True):
    """Elige un listing demo, priorizando uno que sí tenga reseñas."""
    principal_ordered_ids = principal_df[id_col_principal].dropna().astype(str).tolist()

    if prefer_with_reviews:
        for listing_id in principal_ordered_ids:
            if listing_id in ids_with_reviews:
                return listing_id

    if principal_ordered_ids:
        return principal_ordered_ids[0]

    raise ValueError("No hay IDs disponibles en Principal.")


LISTING_ID_DEMO = choose_demo_listing_id(prefer_with_reviews=True)
print("ID demo seleccionado:", LISTING_ID_DEMO)
print("Reseñas del ID demo:", int((reviews_df[id_col_reviews].astype(str) == LISTING_ID_DEMO).sum()))

Vista rápida de Principal:


,Ciudad,Distrito,Fecha de colecta,ID Airbnb,Alias,URL Canónica,Título de cabecera,Título de la propiedad,Resumen de la propiedad,Reconocimiento de Airbnb,Acerca de este espacio,¿Es superhost?,¿Identidad verificada del host?,¿Host tiene foto de perfil?,Tiempo como host en años,Ubicación exacta,Tipo de cama,Número de cuartos o habitaciones,Tipo de alojamiento,Número de servicios o amenidades,Precio por noche,Promedio de reviews o calificaión,Instant bookable,Política de cancelación,Verificar teléfono de huésped,Disponibilidad mayor a 90 días,Número de reviews o reseñas
0,Lima,Barranco,28/10/2024,708973846265169428,Diego Alberto,https://www.airbnb.com.pe/rooms/708973846265169428,"Cálido, céntrico, privado y moderno departamento","Apartamento con servicios incluidos entero en Barranco, Perú",2 huéspedes - 1 habitación - 1 cama - 1 baño,Llegada autónoma\nEl personal del edificio te ayudará a hacer el check-in.\nDiego Alberto tiene la categoría de Supe...,"Moderno (1 año de estreno), céntrico (3 cuadras de supermercados y 2 cuadras de un mercado tradicional) y cálido(Inf...",SI,SI,SI,0.75,SI,1,1,1,28,91,4.88,NO,1,SI,SI,42
1,Lima,Barranco,24/10/2024,1135034995673292208,Patty,https://www.airbnb.com.pe/rooms/1135034995673292208,Studio 1er piso con Jacuzzi privado en Barranco,"Vivienda rentada entero en Barranco, Perú",3 huéspedes - 1 habitación - 2 camas - 1 baño,Zona de trabajo\nUna habitación con wifi que es ideal para trabajar.\nPatty tiene la categoría de Superanfitrión\nLo...,"Descubre este moderno Airbnb en Barranco, a solo 15 min a pie de las mejores playas (los yuyos, las sombrillas). El ...",SI,SI,SI,6,NO,1,1,1,52,106,4.86,NO,2,SI,SI,15
2,Lima,Barranco,28/10/2024,1204320587452540691,Silvia,https://www.airbnb.com.pe/rooms/1204320587452540691,"Artline Barranco 1BR, Pool, Gimnasio, WiFi","Vivienda rentada entero en Lima, Perú",4 huéspedes - 1 habitación - 2 camas - 1 baño,Llegada autónoma\nRealiza la llegada fácilmente mediante la cerradura con teclado.\nUbicación inmejorable\nEn el últ...,"Apartamento con una ubicación privilegiada en el distrito bohemio de Barranco. Cerca de los mejores bares, restauran...",NO,SI,SI,0.25,SI,1,1,1,40,119,5,NO,1,SI,NO,3



Vista rápida de Reviews:


,Fecha de colecta,ID Airbnb,#Review,Reseña (Debe estar solo en español)
0,20/10/2024,981291392324367211,1,.
1,20/10/2024,981291392324367211,2,"Agradable lugar, tiene todo lo necesario."
2,20/10/2024,981291392324367211,3,Muy buena locación y muy agradable el Apto!



Resumen:
Listados en Principal: 52
Listados con reseñas asociadas: 52
Reseñas totales: 2824
ID demo seleccionado: 708973846265169428
Reseñas del ID demo: 42


## Contexto industrial y propósito del módulo LLM/RAG

El módulo LLM forma parte de una aplicación multimodal orientada a una empresa administradora de alojamientos tipo Airbnb en Lima Metropolitana. Su propósito es asistir al equipo comercial y operativo en la consulta rápida de información relevante sobre cada alojamiento, utilizando lenguaje natural.

El chatbot no responde de forma general ni con conocimiento externo. Su funcionamiento está basado en una arquitectura **RAG** (*Retrieval-Augmented Generation*), en la cual el usuario selecciona un alojamiento mediante su `ID Airbnb`, formula una pregunta y el sistema recupera evidencia únicamente de ese listing. La evidencia incluye datos estructurados de la ficha del alojamiento y reseñas asociadas al mismo identificador.

Este diseño permite responder preguntas como: qué aspectos positivos destacan los huéspedes, si existen quejas frecuentes, si la ubicación es una fortaleza, si el alojamiento es adecuado para trabajo remoto o si conviene administrarlo comercialmente. Al limitar la recuperación al alojamiento seleccionado, se reduce el riesgo de mezclar reseñas de distintos anuncios y se mejora la trazabilidad de las respuestas.

## Variables relevantes del módulo LLM

| Variable | Fuente | Utilidad para el chatbot |
|---|---|---|
| `ID Airbnb` | Principal / Reviews | Llave principal para vincular ficha y reseñas del mismo alojamiento. |
| Distrito | Principal | Permite contextualizar geográficamente el alojamiento. |
| Título del alojamiento | Principal | Identifica el listing consultado por el usuario. |
| Tipo de propiedad | Principal | Ayuda a responder sobre la naturaleza del alojamiento. |
| Resumen de la propiedad | Principal | Aporta información sobre capacidad, habitaciones, camas y baños. |
| Precio por noche | Principal | Permite responder consultas relacionadas con valor y conveniencia. |
| Rating promedio | Principal | Aporta una señal cuantitativa de reputación. |
| Host / Superhost | Principal | Sirve como indicador de confianza y experiencia del anfitrión. |
| Amenidades | Principal | Permite responder preguntas sobre comodidad, trabajo remoto o servicios. |
| Reseñas | Reviews | Constituyen la evidencia textual principal para identificar fortalezas, debilidades y experiencia del huésped. |

## Fuentes de datos consideradas

El módulo LLM utiliza directamente dos fuentes principales:

1. **Hoja `Principal`:** contiene la ficha tabular de los alojamientos.
2. **Hoja `Reviews`:** contiene reseñas asociadas por `ID Airbnb`.

En la arquitectura multimodal completa, estas fuentes se complementan con:

3. **Fotografías del alojamiento:** insumo del módulo CNN para evaluar la calidad visual del listing.
4. **Salidas predictivas de MLP/CNN:** predicción tabular de calificación esperada y evaluación visual, usadas posteriormente para fusión multimodal.

Así, la aplicación final integra evidencia **tabular**, **visual** y **textual** para apoyar una decisión comercial.

## Restricciones del entorno y KPIs del módulo LLM

| Restricción | Naturaleza | Impacto en el diseño |
|---|---|---|
| Dependencia de Ollama local | Infraestructura / acceso a tecnología | El sistema requiere que Ollama esté activo y que el modelo local esté instalado. En equipos con menor capacidad, la latencia puede aumentar. |
| Calidad y disponibilidad de reseñas | Datos | Si un alojamiento tiene pocas reseñas o reseñas poco informativas, el chatbot puede no contar con evidencia suficiente para responder con precisión. |

| KPI | Definición | Utilidad |
|---|---|---|
| Tasa de respuestas fundamentadas | Porcentaje de respuestas que utilizan evidencia recuperada de la ficha o reseñas. | Evalúa si el chatbot responde con respaldo documental. |
| Tasa de no alucinación | Porcentaje de respuestas que no incorporan información externa al contexto recuperado. | Evalúa la confiabilidad del módulo LLM/RAG. |
| Tasa de respuesta suficiente | Porcentaje de respuestas que atienden de forma clara la pregunta planteada. | Evalúa utilidad operativa para el usuario final. |

## 6. Catálogo de listados para seleccionar un ID

Esta vista ayuda a elegir un departamento para probar el chatbot.

In [6]:
def build_listing_catalog(principal_df):
    """Construye una tabla resumida de listados para elegir ID."""
    candidate_cols = [id_col_principal]

    preferred_keywords = [
        "titulo", "title", "nombre", "propiedad",
        "precio", "price",
        "calificacion", "rating", "review",
        "superhost", "host",
        "amenidad", "servicio",
        "capacidad", "huesped", "habitacion", "cama", "bano", "baño"
    ]

    for col in principal_df.columns:
        ncol = normalize_name(col)
        if any(normalize_name(k) in ncol for k in preferred_keywords):
            if col not in candidate_cols:
                candidate_cols.append(col)

    return principal_df[candidate_cols].copy()


catalog_df = build_listing_catalog(principal_df)
display(catalog_df.head(15))

,ID Airbnb,Título de cabecera,Título de la propiedad,Resumen de la propiedad,¿Es superhost?,¿Identidad verificada del host?,¿Host tiene foto de perfil?,Tiempo como host en años,Tipo de cama,Número de cuartos o habitaciones,Número de servicios o amenidades,Precio por noche,Promedio de reviews o calificaión,Verificar teléfono de huésped,Número de reviews o reseñas
0,708973846265169428,"Cálido, céntrico, privado y moderno departamento","Apartamento con servicios incluidos entero en Barranco, Perú",2 huéspedes - 1 habitación - 1 cama - 1 baño,SI,SI,SI,0.75,1,1,28,91,4.88,SI,42
1,1135034995673292208,Studio 1er piso con Jacuzzi privado en Barranco,"Vivienda rentada entero en Barranco, Perú",3 huéspedes - 1 habitación - 2 camas - 1 baño,SI,SI,SI,6,1,1,52,106,4.86,SI,15
2,1204320587452540691,"Artline Barranco 1BR, Pool, Gimnasio, WiFi","Vivienda rentada entero en Lima, Perú",4 huéspedes - 1 habitación - 2 camas - 1 baño,NO,SI,SI,0.25,1,1,40,119,5,SI,3
3,899379506224853216,Apt en Barranco con bicicletas-Allin Samay,"Vivienda rentada entero en Barranco, Perú","3 huéspedes - 1 habitación - 2 camas - 1,5 baños",SI,SI,SI,1,1,1,59,119,4.94,SI,32
4,1158688913933082401,Increíble Dpto 1BR cerca a Miraflores | 1310,"Apartamento con servicios incluidos entero en Barranco, Perú",3 huéspedes - 1 habitación - 1 cama - 1 baño,SI,SI,SI,0.5,1,1,32,120,4.76,SI,21
5,871231364866066627,Acogedor condominio 1BDR Barranco Center,"Condo entero en Barranco, Perú",2 huéspedes - 1 habitación - 1 cama - 1 baño,SI,SI,SI,3,1,1,36,123,5,SI,8
6,1259080790992015969,Con aire acondicionado. ¡Perfecto con vistas panorámicas modernas de Lima!,"Vivienda rentada entero en Barranco, Perú",2 huéspedes - 1 habitación - 1 cama - 1 baño,SI,SI,SI,1,1,1,37,130,4.86,SI,5
7,1173484430269756134,"BestView Barranco,Pool, Jacuzzi","Vivienda rentada entero en Barranco, Perú",3 huéspedes - 1 habitación - 1 cama - 1 baño,NO,SI,SI,3,1,1,41,135,4.53,SI,20
8,564368895233936119,w* | Impecable 1BR en Bohem Barranco,"Vivienda rentada entero en Barranco, Perú",2 huéspedes - 1 habitación - 1 cama - 1 baño,NO,SI,SI,5,1,1,34,138,4.73,SI,58
9,1207414356654599414,Exclusivo Apt 1BR con Netflix Gratis | 507,"Vivienda rentada entero en Lima, Perú",4 huéspedes - 1 habitación - 2 camas - 1 baño,SI,SI,SI,0.25,1,1,41,140,4.96,SI,24


## 7. Construcción de documentos del listado

In [7]:
TEXT_FIELD_KEYWORDS = [
    "titulo", "title", "nombre", "cabecera", "propiedad",
    "resumen", "summary", "descripcion", "description", "acerca",
    "espacio", "amenidad", "amenities", "servicio", "services",
    "reconocimiento", "superhost", "host",
    "politica", "cancelacion", "cancelación",
    "precio", "price", "calificacion", "calificación", "rating",
    "review", "reseña", "resena",
    "ubicacion", "ubicación", "distrito", "barrio",
    "capacidad", "huesped", "huésped", "habitacion", "habitación",
    "cama", "bano", "baño", "wifi", "trabajo"
]


def chunk_text(text, max_chars=MAX_CHARS_PER_CHUNK, overlap=CHUNK_OVERLAP):
    """Divide texto largo en chunks con pequeño solapamiento."""
    text = safe_text(text)
    if len(text) <= max_chars:
        return [text] if text else []

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = max(0, end - overlap)
        if start >= len(text):
            break
    return chunks


def get_listing_row(listing_id):
    """Obtiene la fila del listado seleccionado desde Principal."""
    listing_id_str = str(listing_id)
    mask = principal_df[id_col_principal].astype(str) == listing_id_str
    if mask.sum() == 0:
        raise ValueError(f"No se encontró el listing_id {listing_id_str} en Principal.")
    return principal_df.loc[mask].iloc[0]


def get_listing_reviews(listing_id):
    """Obtiene reseñas asociadas al listado seleccionado."""
    listing_id_str = str(listing_id)
    mask = reviews_df[id_col_reviews].astype(str) == listing_id_str
    return reviews_df.loc[mask].copy()


def build_listing_profile_text(row):
    """
    Construye una ficha textual del listado a partir de columnas relevantes.
    Evita depender de nombres exactos, porque el dataset puede cambiar.
    """
    selected_cols = find_columns_by_keywords(principal_df, TEXT_FIELD_KEYWORDS)

    # Si no detecta suficientes columnas, usa todas como fallback.
    if len(selected_cols) < 5:
        selected_cols = list(principal_df.columns)

    lines = []
    lines.append("FICHA DEL LISTADO")
    lines.append(f"{id_col_principal}: {safe_text(row[id_col_principal])}")

    for col in selected_cols:
        value = safe_text(row.get(col, ""))
        if value and value.lower() not in ["nan", "none"]:
            lines.append(f"{col}: {value}")

    return "\n".join(lines)


def build_documents(listing_id, max_reviews=MAX_REVIEWS_PER_LISTING):
    """
    Construye documentos del listado:
    1. ficha estructurada desde Principal;
    2. reseñas individuales desde Reviews.
    """
    row = get_listing_row(listing_id)
    listing_reviews = get_listing_reviews(listing_id)

    documents = []

    # Documento base: ficha del listado
    profile_text = build_listing_profile_text(row)
    for i, ch in enumerate(chunk_text(profile_text)):
        documents.append({
            "listing_id": str(listing_id),
            "source": "ficha_principal",
            "source_id": f"ficha_{i+1}",
            "text": ch
        })

    # Documentos de reseñas
    if len(listing_reviews) > 0:
        # Limita reseñas por eficiencia del MVP
        listing_reviews = listing_reviews.head(max_reviews).copy()

        for idx, r in listing_reviews.iterrows():
            review_text = safe_text(r.get(review_text_col, ""))
            if not review_text:
                continue

            # Si existe una columna #Review o similar, úsala como referencia.
            review_id_col = find_column(
                listing_reviews,
                ["#Review", "Review ID", "review_id", "id review"],
                required=False
            )
            review_ref = safe_text(r.get(review_id_col, idx)) if review_id_col else str(idx)

            for j, ch in enumerate(chunk_text(review_text)):
                documents.append({
                    "listing_id": str(listing_id),
                    "source": "reseña",
                    "source_id": f"review_{review_ref}_chunk_{j+1}",
                    "text": ch
                })

    return documents


# Prueba rápida con el primer ID disponible
LISTING_ID_DEMO = choose_demo_listing_id(prefer_with_reviews=True)
docs_demo = build_documents(LISTING_ID_DEMO)

print("ID de prueba:", LISTING_ID_DEMO)
print("Documentos construidos:", len(docs_demo))
print("\nPrimer documento:")
print(docs_demo[0]["text"][:1200])

ID de prueba: 708973846265169428
Documentos construidos: 44

Primer documento:
FICHA DEL LISTADO ID Airbnb: 708973846265169428 Distrito: Barranco Título de cabecera: Cálido, céntrico, privado y moderno departamento Título de la propiedad: Apartamento con servicios incluidos entero en Barranco, Perú Resumen de la propiedad: 2 huéspedes - 1 habitación - 1 cama - 1 baño Reconocimiento de Airbnb: Llegada autónoma El personal del edificio te ayudará a hacer el check-in. Diego Alberto tiene la categoría de Superanfitrión Los Superanfitriones son anfitriones experimentados, con evaluaciones excelentes. Las mascotas son bienvenidas Lleva a tus mascotas al alojamiento. Acerca de este espacio: Moderno (1 año de estreno), céntrico (3 cuadras de supermercados y 2 cuadras de un mercado tradicional) y cálido(Infraestructura hogareña) Cuenta con: Juegos de mesa, Streaming (Netflix), Música (Spotify), Soporte tecnológico (Alexa) Adicionales: Cervezas Premium (En la nevera), Cocher


## 8. Recuperación de contexto relevante

El recuperador opera en modo `auto`: intenta usar embeddings multilingües y, si el entorno local presenta problemas con `sentence-transformers` o PyTorch, activa TF-IDF como respaldo estable. Esto permite mantener el flujo RAG operativo con Ollama.


In [8]:
SPANISH_STOPWORDS = [
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuando",
    "de", "del", "desde", "donde", "durante", "e", "el", "ella", "ellas", "ellos",
    "en", "entre", "era", "es", "esa", "esas", "ese", "eso", "esos", "esta",
    "estaba", "estado", "estan", "están", "este", "esto", "estos", "fue", "ha",
    "hay", "la", "las", "le", "les", "lo", "los", "mas", "más", "me", "mi",
    "mis", "muy", "no", "nos", "o", "para", "pero", "por", "porque", "que",
    "qué", "se", "si", "sí", "sin", "sobre", "su", "sus", "tambien", "también",
    "te", "tiene", "un", "una", "uno", "y", "ya"
]

_embedding_model_cache = None
_embedding_error_message = None


def expand_question(question):
    """
    Expande preguntas frecuentes del negocio con términos esperables en reseñas.
    Esto ayuda tanto a TF-IDF como a embeddings cuando la pregunta es genérica.
    """
    q = normalize_name(question)
    additions = []

    if any(w in q for w in ["positivo", "destacan", "recomiendan", "fortaleza", "bueno"]):
        additions.append("limpio cómodo bonito ordenado recomendado agradable tranquilo céntrico ubicación host atento comunicación excelente")

    if any(w in q for w in ["queja", "quejas", "mejorar", "debilidad", "riesgo", "problema", "malo", "negativo"]):
        additions.append("ruido sucio limpieza privacidad demora check in problema incómodo lejos atención mantenimiento vecinos")

    if any(w in q for w in ["ubicacion", "ubicación", "zona", "barrio", "cerca", "centrico", "céntrico"]):
        additions.append("ubicación céntrico cerca zona barrio tranquilo seguro restaurantes supermercados mercado playa transporte")

    if any(w in q for w in ["precio", "valor", "razonable", "calidad", "caro", "barato"]):
        additions.append("precio calidad relación precio calidad valor costo cumple descripción fotos recomendado")

    if any(w in q for w in ["trabajo", "remoto", "wifi", "internet", "teletrabajo"]):
        additions.append("wifi internet escritorio trabajar remoto tranquilo ruido comodidad streaming soporte tecnológico alexa")

    if any(w in q for w in ["familia", "grupo", "huespedes", "huéspedes", "ninos", "niños"]):
        additions.append("huéspedes habitación cama baño familia grupo espacio cómodo mascotas")

    if additions:
        return question + " " + " ".join(additions)

    return question


def load_embedding_model():
    """
    Carga una sola vez el modelo de embeddings multilingüe.
    Si falla en Windows por DLL/PyTorch, no detiene el notebook: activa fallback TF-IDF.
    """
    global _embedding_model_cache, _embedding_error_message

    if _embedding_model_cache is None:
        try:
            from sentence_transformers import SentenceTransformer
            _embedding_model_cache = SentenceTransformer(EMBEDDING_MODEL_NAME)
            _embedding_error_message = None
        except Exception as e:
            _embedding_error_message = repr(e)
            print("Aviso: no se pudo cargar sentence-transformers en este entorno local.")
            print("Se usará TF-IDF como fallback estable para la demo con Ollama.")
            print("Detalle técnico:", _embedding_error_message)
            _embedding_model_cache = False

    return _embedding_model_cache


def retrieve_context_tfidf(question, documents, top_k=TOP_K, min_score=0.0):
    """Recuperador baseline estable: TF-IDF + similitud coseno."""
    question_expanded = expand_question(question)

    texts = [d["text"] for d in documents]
    corpus = texts + [question_expanded]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        stop_words=SPANISH_STOPWORDS,
        max_features=6000
    )

    tfidf = vectorizer.fit_transform(corpus)
    doc_vectors = tfidf[:-1]
    query_vector = tfidf[-1]

    scores = cosine_similarity(query_vector, doc_vectors).flatten()
    ranked_idx = np.argsort(scores)[::-1]

    results = []
    for idx in ranked_idx:
        if len(results) >= top_k:
            break
        if scores[idx] >= min_score:
            d = documents[idx].copy()
            d["score"] = float(scores[idx])
            d["retriever"] = "tfidf"
            results.append(d)

    return results


def retrieve_context_embeddings(question, documents, top_k=TOP_K):
    """
    Recuperador semántico: embeddings multilingües + similitud coseno.
    Si sentence-transformers falla, usa TF-IDF sin detener el pipeline.
    """
    model = load_embedding_model()
    if model is False:
        results = retrieve_context_tfidf(question, documents, top_k=top_k)
        for d in results:
            d["retriever"] = "tfidf_fallback_por_error_embeddings"
        return results

    question_expanded = expand_question(question)
    texts = [d["text"] for d in documents]

    doc_embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    query_embedding = model.encode([question_expanded], normalize_embeddings=True, show_progress_bar=False)

    scores = cosine_similarity(query_embedding, doc_embeddings).flatten()
    ranked_idx = np.argsort(scores)[::-1]

    results = []
    for idx in ranked_idx[:top_k]:
        d = documents[idx].copy()
        d["score"] = float(scores[idx])
        d["retriever"] = "embeddings"
        results.append(d)

    return results


def retrieve_context(question, documents, top_k=TOP_K, min_score=0.0, always_include_profile=True, mode=RETRIEVER_MODE):
    """
    Recupera los documentos más relevantes para una pregunta.

    Modos:
    - auto: intenta embeddings; si falla, usa TF-IDF.
    - embeddings: intenta recuperación semántica; si falla, usa fallback TF-IDF.
    - tfidf: fuerza TF-IDF, estable para demo local.
    """
    question = safe_text(question)
    if not question or not documents:
        return []

    profile_docs = [d for d in documents if d["source"] == "ficha_principal"]
    search_docs = documents

    mode = str(mode).lower().strip()

    if mode in ["auto", "embeddings"]:
        results = retrieve_context_embeddings(question, search_docs, top_k=top_k)
    elif mode == "tfidf":
        results = retrieve_context_tfidf(question, search_docs, top_k=top_k, min_score=min_score)
    else:
        raise ValueError("RETRIEVER_MODE debe ser 'auto', 'embeddings' o 'tfidf'.")

    # Incluye la ficha base si no salió entre los primeros resultados.
    if always_include_profile and profile_docs and not any(r["source"] == "ficha_principal" for r in results):
        profile_doc = profile_docs[0].copy()
        profile_doc["score"] = None
        profile_doc["retriever"] = "contexto_fijo"
        results.insert(0, profile_doc)

    return results


# Prueba del recuperador
question_demo = "¿Qué aspectos positivos destacan los huéspedes?"
retrieved_demo = retrieve_context(question_demo, docs_demo, top_k=5)

retrievers_used = sorted(set(d.get("retriever", "") for d in retrieved_demo))
print("Modo configurado:", RETRIEVER_MODE)
print("Recuperadores usados en esta ejecución:", retrievers_used)

for i, d in enumerate(retrieved_demo, start=1):
    print("=" * 90)
    print(f"Fuente {i}: {d['source']} | {d['source_id']} | score={d.get('score')} | retriever={d.get('retriever')}")
    print(d["text"][:700])

Aviso: no se pudo cargar sentence-transformers en este entorno local.
Se usará TF-IDF como fallback estable para la demo con Ollama.
Detalle técnico: OSError(22, 'A dynamic link library (DLL) initialization routine failed.', None, 1114)
Modo configurado: auto
Recuperadores usados en esta ejecución: ['tfidf_fallback_por_error_embeddings']
Fuente 1: ficha_principal | ficha_1 | score=0.1151298490175437 | retriever=tfidf_fallback_por_error_embeddings
FICHA DEL LISTADO ID Airbnb: 708973846265169428 Distrito: Barranco Título de cabecera: Cálido, céntrico, privado y moderno departamento Título de la propiedad: Apartamento con servicios incluidos entero en Barranco, Perú Resumen de la propiedad: 2 huéspedes - 1 habitación - 1 cama - 1 baño Reconocimiento de Airbnb: Llegada autónoma El personal del edificio te ayudará a hacer el check-in. Diego Alberto tiene la categoría de Superanfitrión Los Superanfitriones son anfitriones experimentados, con evaluaciones excelentes. Las mascotas son bienveni

## 8.1 Reglas actuales del prompt del chatbot en el aplicativo

### Reglas generales

1. Responder únicamente a la pregunta actual del usuario.
2. No agregar comentarios sobre temas no solicitados.
3. Usar la intención detectada para decidir qué fuentes son relevantes.
4. No mencionar ausencia de mejoras, quejas o problemas si la pregunta no trata sobre eso.
5. Usar únicamente las fuentes listadas en el prompt.
6. Si una fuente no aporta evidencia útil para la intención, no mencionarla artificialmente.
7. La ficha del anuncio se usa para datos objetivos: capacidad, habitaciones, camas, baños, precio, rating, host, superhost, amenidades, distrito y descripción objetiva.
8. Las reseñas se usan para experiencia del huésped: comodidad, limpieza, ubicación percibida, trato del anfitrión, comunicación, quejas, relación precio-calidad, fotos, recomendación y adecuación para tipos de huésped.
9. Si ficha y reseñas aportan, sintetizar ambas.
10. Si solo una fuente aporta, responder solo con esa fuente.
11. Si hay varias reseñas relevantes, no basarse en una sola.
12. Resumir patrones comunes y matices importantes.
13. El score de recuperación no decide por sí solo qué se menciona.
14. Considerar frases directamente relacionadas aunque tengan menor score.
15. Si varias reseñas coinciden, usar formulaciones como: “Varias reseñas destacan”, “Los huéspedes coinciden en” o “Se repite como fortaleza”.
16. Si solo una reseña aporta información real, aclarar que la evidencia textual es limitada.
17. Si no hay evidencia suficiente, responder: “No hay evidencia suficiente en la ficha o reseñas recuperadas para afirmarlo con seguridad.”
18. No mencionar relevancia, score, similitud, porcentajes de recuperación ni metadatos técnicos.
19. Responder en un párrafo breve y natural.
20. No agregar una sección de evidencia dentro de la respuesta, porque la interfaz ya muestra las fuentes.

### Reglas por intención

- Mejoras / quejas / problemas: solo mencionar mejoras si hay críticas claras.
- Capacidad: usar capacidad, huéspedes, habitaciones, camas, baños y reseñas directamente relacionadas.
- Quejas: usar críticas claras; si hay una crítica aislada, aclarar que no parece frecuente.
- Precio / valor: combinar precio, rating o amenidades de ficha con reseñas relevantes, solo si existen.
- Limpieza: usar reseñas sobre limpieza, orden o higiene.
- Ubicación: usar distrito, descripción objetiva y reseñas sobre zona, cercanía, acceso o restaurantes.
- Anfitrión: usar reseñas sobre trato, comunicación, cordialidad o atención.
- Amenidades / servicios: usar descripción objetiva, amenidades y reseñas que mencionen la amenidad; no usar capacidad salvo que el usuario lo pregunte.
- Conveniencia comercial: integrar ficha y reseñas para resumir fortalezas, riesgos y señales de decisión.


## 8.2 Selección de evidencia alineada a la app

La app no envía al LLM todo lo recuperado. Primero detecta la intención de la pregunta y luego decide qué datos de ficha y qué reseñas son útiles.

La salida se separa en facts, used_reviews, unused_reviews y prompt_docs. Si no hay reseñas usadas, el bloque de candidatas no usadas no se muestra.


In [ ]:
INTENT_KEYWORDS = {
    "amenidades": ["wifi", "wi fi", "wi-fi", "internet", "piscina", "pool", "jacuzzi", "gimnasio", "gym", "coworking", "cowork", "aire acondicionado", "cocina", "lavadora", "secadora", "estacionamiento", "balcón", "televisión", "tv"],
    "capacidad": ["capacidad", "personas", "persona", "huéspedes", "huespedes", "familia", "grupo", "habitaciones", "habitación", "camas", "baños"],
    "limpieza": ["limpieza", "limpio", "limpia", "ordenado", "impecable"],
    "ubicacion": ["ubicación", "ubicacion", "zona", "cerca", "restaurantes", "cafeterías", "barranco"],
    "anfitrion": ["anfitrión", "anfitrion", "host", "comunicación", "atención", "amable", "responde"],
    "precio": ["precio", "calidad", "valor", "caro", "barato"],
    "mejoras": ["mejorar", "debería mejorar", "debilidades", "problemas", "quejas", "aspectos negativos"],
    "quejas": ["queja", "quejas", "problema", "críticas", "negativo"],
    "comercial": ["conviene administrar", "decisión comercial", "administrar", "rentable"],
}
CAPACITY_REVIEW_TERMS = ["pareja", "dos personas", "2 personas", "persona", "personas", "huésped", "huéspedes", "familia", "grupo", "cómodo", "acogedor", "equipado", "estadía", "estancia"]

def normalize_query(text):
    text = safe_text(text).lower()
    text = unidecode(text)
    text = re.sub(r"\bwi[\s-]*fi\b", "wifi", text)
    text = re.sub(r"[^a-z0-9ñ\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def contains_any_term(text, terms):
    nt = normalize_query(text)
    return any(normalize_query(term) in nt for term in terms)

def detect_question_intent(question):
    q = normalize_query(question)
    categories = [cat for cat, terms in INTENT_KEYWORDS.items() if any(normalize_query(t) in q for t in terms)]
    if not categories:
        categories = ["general"]
    amenity_terms = [t for t in INTENT_KEYWORDS["amenidades"] if normalize_query(t) in q]
    amenity_terms = sorted(set(["wifi" if t in ["wi fi", "wi-fi", "internet"] else "piscina" if t == "pool" else t for t in amenity_terms]))
    return {"categories": categories, "amenity_terms": amenity_terms, "asks_about_amenities": "amenidades" in categories, "asks_about_capacity": "capacidad" in categories, "asks_about_improvements": "mejoras" in categories, "asks_about_complaints": "quejas" in categories, "asks_commercial_decision": "comercial" in categories}

def is_review_doc(doc):
    return doc.get("source") == "reseña"

def is_profile_doc(doc):
    return doc.get("source") == "ficha_principal"

def select_rag_evidence(question, documents, top_k=TOP_K):
    intent = detect_question_intent(question)
    raw = retrieve_context(question, documents, top_k=max(MAX_CANDIDATES, top_k), always_include_profile=False, mode=RETRIEVER_MODE)
    reviews = [d for d in raw if is_review_doc(d)]
    def supports(doc):
        text = doc.get("text", "")
        cats = intent["categories"]
        if intent["asks_about_amenities"]:
            return contains_any_term(text, intent["amenity_terms"] or INTENT_KEYWORDS["amenidades"])
        if intent["asks_about_capacity"]:
            return contains_any_term(text, CAPACITY_REVIEW_TERMS)
        for cat in ["limpieza", "ubicacion", "anfitrion", "precio", "mejoras", "quejas"]:
            if cat in cats:
                return contains_any_term(text, INTENT_KEYWORDS[cat])
        return True
    used_reviews = [d for d in reviews if supports(d)][:MAX_USED_REVIEWS]
    used_ids = {d.get("source_id") for d in used_reviews}
    unused_reviews = [d for d in reviews if d.get("source_id") not in used_ids][:MAX_USED_REVIEWS] if used_reviews else []
    profile_docs = [d for d in documents if is_profile_doc(d)]
    facts = profile_docs[:1] if not intent["asks_about_amenities"] else [d for d in profile_docs if contains_any_term(d.get("text", ""), intent["amenity_terms"] or [])][:1]
    prompt_docs = facts + used_reviews
    return {"intent": intent, "facts": facts, "used_reviews": used_reviews, "unused_reviews": unused_reviews, "prompt_docs": prompt_docs, "candidate_count": len(reviews)}

selection_demo = select_rag_evidence("¿Tiene wi fi el alojamiento?", docs_demo, top_k=5)
print("Intención detectada:", selection_demo["intent"])
print("Reseñas usadas:", len(selection_demo["used_reviews"]))


## Arquitectura del módulo LLM/RAG

El módulo LLM se implementa mediante una arquitectura RAG conectada a Ollama. El sistema primero recupera evidencia del alojamiento seleccionado y luego construye un prompt controlado para que el modelo responda únicamente con base en dicha evidencia.

```text
Usuario selecciona ID Airbnb
        ↓
Se carga ficha del listado desde Principal
        ↓
Se cargan reseñas asociadas desde Reviews
        ↓
Se construyen documentos/chunks con metadata
        ↓
Se recuperan evidencias relevantes
        ↓
Se construye prompt controlado
        ↓
Ollama genera respuesta
        ↓
Se muestra respuesta + fuentes usadas
```

### Capas conceptuales del LLM usado vía Ollama

Aunque el modelo no se entrena desde cero en este trabajo, se documenta su estructura conceptual para cumplir con el diseño arquitectónico solicitado.

| Componente / capa | Descripción | Activación o función asociada |
|---|---|---|
| Entrada | Tokenización del prompt compuesto por pregunta, ficha y reseñas recuperadas. | No aplica directamente. |
| Embeddings | Representación vectorial de los tokens de entrada. | Transformación densa aprendida. |
| Bloques Transformer | Capas internas de atención que modelan relaciones entre tokens del contexto. | Atención, normalización y conexiones residuales. |
| Feed Forward | Subcapas densas internas del Transformer. | GELU o SwiGLU, según arquitectura del modelo. |
| Salida | Generación autoregresiva de tokens de respuesta. | Softmax sobre el vocabulario. |

## Integración con la arquitectura multimodal

El módulo LLM/RAG se integra con los demás componentes de la aplicación de la siguiente manera:

- **MLP:** procesa variables tabulares del alojamiento y predice la calificación esperada.
- **CNN:** procesa imágenes del alojamiento y estima la calidad visual o categoría visual del listing.
- **LLM/RAG:** interpreta la ficha y las reseñas para responder consultas en lenguaje natural y generar explicación textual.

### Estrategias de combinación propuestas

| Estrategia | Descripción | Uso previsto |
|---|---|---|
| Fusión tardía (*late fusion*) | Combina salidas independientes de MLP, CNN y LLM mediante reglas o ponderaciones. | Baseline interpretable para generar un score final. |
| Orquestación con LLM | El LLM usa las salidas de MLP y CNN junto con evidencia textual para explicar la recomendación. | Capa explicativa orientada al usuario final. |

### Propuesta de escalabilidad y GPU

- El módulo CNN se beneficia directamente de GPU para inferencia visual.
- El módulo LLM puede ejecutarse localmente con Ollama en equipos con GPU, reduciendo dependencia de APIs externas.
- Para una versión productiva, se recomienda separar servicios: API de inferencia visual, API tabular, API RAG/LLM y frontend web.
- Si aumenta el número de listados o consultas concurrentes, se recomienda cachear embeddings/evidencias y asignar GPU prioritaria a CNN/LLM.

## Comparación funcional dentro del módulo LLM

Aunque la rúbrica exige comparación cuantitativa principalmente para MLP y CNN, el módulo LLM también considera variantes operativas para justificar su diseño.

| Variante | Ventajas | Desventajas | Uso en el notebook |
|---|---|---|---|
| TF-IDF + Ollama | Estable, rápido, no depende de PyTorch; útil como fallback local. | Menor comprensión semántica; depende de coincidencia de términos. | Respaldo automático si embeddings falla. |
| Embeddings multilingües + Ollama | Mejor recuperación semántica en español; útil para preguntas generales. | Puede fallar en algunos entornos Windows por dependencias de `sentence-transformers`/PyTorch. | Modo preferente cuando el entorno lo permite. |
| Fallback extractivo sin LLM | Permite responder si Ollama no está disponible. | No genera una respuesta tan natural ni explicativa. | Modo de contingencia. |
| RAG + Ollama | Genera respuestas naturales con evidencia recuperada. | Requiere Ollama activo y modelo descargado. | Modo principal del módulo LLM. |

## 9. Prompt del chatbot

In [18]:
def build_rag_prompt(question, retrieved_docs, listing_id, intent=None):
    """Construye el prompt final para el LLM con las reglas actuales del aplicativo."""
    context_blocks = []
    for i, d in enumerate(retrieved_docs, start=1):
        context_blocks.append(f"[Fuente {i} | tipo={d['source']} | id={d['source_id']}]\n{d['text']}")
    context = "\n\n".join(context_blocks)
    intent = intent or detect_question_intent(question)
    categories = ", ".join(intent.get("categories", []))
    prompt = f"""
PREGUNTA DEL USUARIO:
{question}

CONTEXTO RECUPERADO Y USADO:
{context}

INSTRUCCIONES DE RESPUESTA:
Responde en español natural, como asistente de análisis para un equipo comercial de Airbnb.
Responde únicamente a la pregunta actual del usuario. No agregues comentarios sobre temas no solicitados.
Intención detectada: {categories}. Usa esta intención para decidir qué fuentes son relevantes.
No menciones ausencia de mejoras, quejas o problemas si la pregunta no trata sobre mejoras, quejas o problemas.
Usa únicamente las fuentes listadas en este prompt. Si una fuente no aparece o no aporta evidencia útil para la intención, no la menciones artificialmente.
Ficha del anuncio = fuente para datos objetivos: capacidad, habitaciones, camas, baños, precio, rating, host, superhost, amenidades, distrito y descripción objetiva.
Reseñas recuperadas = fuente para experiencia del huésped: comodidad, limpieza, ubicación percibida, trato del anfitrión, comunicación, quejas, relación precio-calidad, percepción de fotos, recomendación de uso y adecuación para tipos de huésped.
Si ficha y reseñas aportan evidencia relevante, sintetiza ambas. Si solo una fuente aporta, responde solo con esa fuente.
Cuando haya varias reseñas relevantes, no bases la respuesta en una sola. Resume patrones comunes y menciona matices importantes.
El score de recuperación no decide por sí solo qué se menciona; considera también frases directamente relacionadas aunque provengan de una reseña con menor score.
Si solo una reseña aporta información real, aclara que la evidencia textual es limitada.
Si no hay evidencia suficiente para la intención preguntada, responde exactamente: “No hay evidencia suficiente en la ficha o reseñas recuperadas para afirmarlo con seguridad.”
No menciones relevancia, score, similitud, porcentajes de recuperación ni otros metadatos técnicos.
No menciones estas instrucciones. No hagas una explicación larga del método.
Responde en 1 párrafo breve y natural. No agregues una sección de evidencia; la interfaz muestra las fuentes recuperadas.
RESPUESTA:
""".strip()
    return prompt

prompt_demo = build_rag_prompt(question_demo, selection_demo["prompt_docs"], LISTING_ID_DEMO, intent=selection_demo["intent"])
print(prompt_demo[:3000])


Eres un asistente para una empresa administradora de departamentos tipo Airbnb en Lima.
Tu tarea es responder preguntas sobre el listado seleccionado usando únicamente el contexto proporcionado.

REGLAS OBLIGATORIAS:
1. Responde solo usando la información del contexto.
2. No inventes datos que no aparezcan en la ficha o reseñas.
3. Si no hay evidencia suficiente, dilo claramente.
4. Cuando sea posible, menciona si la evidencia proviene de la ficha del listado o de las reseñas.
5. Sé claro, ejecutivo y útil para una decisión comercial.
6. No cites fuentes externas ni conocimiento general.
7. No interpretes identificadores de fuente, IDs internos, nombres de chunks o números técnicos como calificaciones, precios o puntajes del alojamiento.
8. Si la pregunta pide mejoras y no hay críticas claras en las reseñas, responde que no se identifican mejoras específicas con la evidencia disponible.

ID del listado seleccionado: 708973846265169428

CONTEXTO RECUPERADO:
[Fuente 1 | tipo=ficha_princi

## 10. Conexión opcional con un LLM

Para el MVP, puedes validar la recuperación usando `LLM_PROVIDER = "none"`.  
Cuando quieras generar respuestas naturales, configura uno de estos proveedores:

- `"gemini"`: usando `GEMINI_API_KEY`.
- `"openai"`: usando `OPENAI_API_KEY`.
- `"ollama"`: para ejecución local con Ollama, más útil cuando pasen a Streamlit/local.

In [19]:
def fallback_without_llm(prompt, retrieved_docs):
    """
    Fallback cuando aún no se configuró un proveedor LLM.
    No simula inteligencia generativa: muestra evidencia recuperada y una respuesta extractiva básica.
    Sirve para validar que el RAG está recuperando información pertinente.
    """
    if not retrieved_docs:
        return "No hay evidencia suficiente en la ficha o reseñas del listado seleccionado."

    evidence_preview = []
    for i, d in enumerate(retrieved_docs[:3], start=1):
        txt = d["text"]
        txt = txt[:500] + ("..." if len(txt) > 500 else "")
        evidence_preview.append(f"Fuente {i} ({d['source']}): {txt}")

    return (
        "Modo MVP sin LLM configurado.\n\n"
        "Evidencia recuperada para responder la pregunta:\n\n"
        + "\n\n".join(evidence_preview)
        + "\n\nPara obtener una respuesta redactada tipo chatbot, configura LLM_PROVIDER = 'gemini', 'openai' u 'ollama'."
    )


def call_llm(prompt, provider=LLM_PROVIDER, retrieved_docs=None):
    """
    Llama al proveedor LLM seleccionado.
    Si provider='none', devuelve una respuesta extractiva de validación.
    """
    provider = provider.lower().strip()

    if provider == "none":
        return fallback_without_llm(prompt, retrieved_docs or [])

    if provider == "gemini":
        import google.generativeai as genai
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("Falta configurar GEMINI_API_KEY en variables de entorno.")
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        response = model.generate_content(prompt)
        return response.text

    if provider == "openai":
        from openai import OpenAI
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("Falta configurar OPENAI_API_KEY en variables de entorno.")
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "Responde únicamente con base en el contexto proporcionado."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content

    if provider == "ollama":
        import requests
        payload = {
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2
            }
        }
        response = requests.post(OLLAMA_URL, json=payload, timeout=180)
        response.raise_for_status()
        return response.json().get("response", "")

    raise ValueError("Proveedor no reconocido. Usa: 'none', 'gemini', 'openai' u 'ollama'.")

## 11. Función principal del chatbot

In [20]:
def answer_question(listing_id, question, top_k=TOP_K, provider=LLM_PROVIDER, show_prompt=False):
    """
    Pipeline completo:
    1. Construye documentos del listing.
    2. Recupera contexto relevante.
    3. Construye prompt.
    4. Genera respuesta con LLM o fallback.
    """
    documents = build_documents(listing_id)

    if not documents:
        return {
            "listing_id": str(listing_id),
            "question": question,
            "answer": "No hay información suficiente para este listado.",
            "retrieved_docs": [],
            "prompt": ""
        }

    retrieved_docs = retrieve_context(question, documents, top_k=top_k)

    if not retrieved_docs:
        return {
            "listing_id": str(listing_id),
            "question": question,
            "answer": "No hay evidencia suficiente en la ficha o reseñas del listado seleccionado.",
            "retrieved_docs": [],
            "prompt": ""
        }

    prompt = build_rag_prompt(question, retrieved_docs, listing_id)

    if show_prompt:
        print(prompt)

    answer = call_llm(prompt, provider=provider, retrieved_docs=retrieved_docs)

    return {
        "listing_id": str(listing_id),
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt
    }


def display_chatbot_result(result):
    """Muestra la respuesta y las evidencias en formato legible."""
    display(Markdown(f"### Pregunta\n{result['question']}"))
    display(Markdown(f"### Respuesta\n{result['answer']}"))

    if result["retrieved_docs"]:
        display(Markdown("### Evidencia recuperada"))
        for i, d in enumerate(result["retrieved_docs"], start=1):
            score = d.get("score")
            score_txt = "ficha base" if score is None else f"{score:.3f}"
            retriever_txt = d.get("retriever", "")
            preview = d["text"][:900] + ("..." if len(d["text"]) > 900 else "")
            display(Markdown(
                f"**Fuente {i}:** `{d['source']}` — `{d['source_id']}` — score: `{score_txt}` — retriever: `{retriever_txt}`\n\n"
                f"> {preview}"
            ))

## 12. Demo rápida del chatbot

In [21]:
# Selecciona un ID de prueba.
# Puedes cambiarlo por cualquier ID visible en catalog_df.
LISTING_ID_DEMO = choose_demo_listing_id(prefer_with_reviews=True)

question_demo = "¿Qué aspectos positivos destacan los huéspedes?"

result_demo = answer_question(
    listing_id=LISTING_ID_DEMO,
    question=question_demo,
    top_k=5,
    provider=LLM_PROVIDER
)

display_chatbot_result(result_demo)

### Pregunta
¿Qué aspectos positivos destacan los huéspedes?

### Respuesta
Según las reseñas disponibles, los huéspedes destacan la tranquilidad del lugar (review_1576_chunk_1), la buena ubicación (review_1576_chunk_1 y review_1569_chunk_1), el ambiente cómodo y limpio (review_1570_chunk_1), la organización y limpieza del departamento (review_1589_chunk_1) y el servicio excelente (review_1570_chunk_1 y review_1569_chunk_1).

### Evidencia recuperada

**Fuente 1:** `ficha_principal` — `ficha_1` — score: `0.115` — retriever: `tfidf_fallback_por_error_embeddings`

> FICHA DEL LISTADO ID Airbnb: 708973846265169428 Distrito: Barranco Título de cabecera: Cálido, céntrico, privado y moderno departamento Título de la propiedad: Apartamento con servicios incluidos entero en Barranco, Perú Resumen de la propiedad: 2 huéspedes - 1 habitación - 1 cama - 1 baño Reconocimiento de Airbnb: Llegada autónoma El personal del edificio te ayudará a hacer el check-in. Diego Alberto tiene la categoría de Superanfitrión Los Superanfitriones son anfitriones experimentados, con evaluaciones excelentes. Las mascotas son bienvenidas Lleva a tus mascotas al alojamiento. Acerca de este espacio: Moderno (1 año de estreno), céntrico (3 cuadras de supermercados y 2 cuadras de un mercado tradicional) y cálido(Infraestructura hogareña) Cuenta con: Juegos de mesa, Streaming (Netflix), Música (Spotify), Soporte tecnológico (Alexa) Adicionales: Cervezas Premium (En la nevera), Cocher

**Fuente 2:** `reseña` — `review_1576_chunk_1` — score: `0.092` — retriever: `tfidf_fallback_por_error_embeddings`

> 100% recomendado, lugar tranquilo con muy buena ubicación.

**Fuente 3:** `reseña` — `review_1570_chunk_1` — score: `0.091` — retriever: `tfidf_fallback_por_error_embeddings`

> Excelente lugar, muy cómodo, limpio y organizado, tuve uane estadía muy agradable, excelente servicio

**Fuente 4:** `reseña` — `review_1589_chunk_1` — score: `0.085` — retriever: `tfidf_fallback_por_error_embeddings`

> El departamente muy bonito, limpio y ordenado. Lo recomiendo.

**Fuente 5:** `reseña` — `review_1569_chunk_1` — score: `0.075` — retriever: `tfidf_fallback_por_error_embeddings`

> Excelente ambiente, servicio y buena ubicación recomendado al 100

## 13. Demo interactiva en Colab

In [ ]:
import ipywidgets as widgets

listing_options = principal_df[id_col_principal].dropna().astype(str).unique().tolist()

listing_dropdown = widgets.Dropdown(
    options=listing_options,
    value=listing_options[0] if listing_options else None,
    description="Listing ID:",
    layout=widgets.Layout(width="70%")
)

question_box = widgets.Textarea(
    value="¿El departamento es bueno para trabajar remoto?",
    description="Pregunta:",
    layout=widgets.Layout(width="90%", height="90px")
)

provider_dropdown = widgets.Dropdown(
    options=["none", "gemini", "openai", "ollama"],
    value=LLM_PROVIDER,
    description="LLM:",
    layout=widgets.Layout(width="50%")
)

run_button = widgets.Button(
    description="Responder",
    button_style="primary"
)

output_area = widgets.Output()


def on_run_button_clicked(b):
    with output_area:
        output_area.clear_output()
        try:
            result = answer_question(
                listing_id=listing_dropdown.value,
                question=question_box.value,
                top_k=TOP_K,
                provider=provider_dropdown.value
            )
            display_chatbot_result(result)
        except Exception as e:
            print("Error:", repr(e))


run_button.on_click(on_run_button_clicked)

display(listing_dropdown, question_box, provider_dropdown, run_button, output_area)

## Evaluación del módulo LLM/RAG

La evaluación del chatbot se realiza mediante preguntas de prueba aplicadas a un alojamiento seleccionado. Para cada pregunta se registra la respuesta generada, las evidencias recuperadas y una evaluación automática basada en reglas.

Debido a que un LLM puede producir variaciones leves entre ejecuciones, la evaluación no se basa en coincidencia textual exacta ni en una calificación manual corrida por corrida. En su lugar, se evalúa si la respuesta:

| Criterio | Descripción |
|---|---|
| Evidencia correcta | La respuesta cuenta con evidencia recuperada desde ficha o reseñas del listing seleccionado. |
| Respuesta suficiente | La respuesta atiende la intención de la pregunta del usuario. |
| No alucinación | La respuesta evita afirmaciones no sustentadas por el contexto recuperado o declara falta de evidencia cuando corresponde. |

Esta evaluación complementa la evaluación cuantitativa de los módulos MLP y CNN, y permite verificar la confiabilidad del componente LLM dentro de la aplicación multimodal.


## 14. Preguntas de prueba para evaluación del módulo LLM

In [22]:
TEST_QUESTIONS = [
    "¿El departamento es bueno para trabajar remoto?",
    "¿Hay quejas frecuentes en las reseñas?",
    "¿Qué aspectos positivos destacan los huéspedes?",
    "¿El precio parece razonable para lo que ofrece?",
    "¿Qué debería mejorar el host?",
    "¿La ubicación parece ser una fortaleza del listado?",
    "¿El alojamiento parece adecuado para familias o grupos?",
    "¿Conviene administrar este departamento según la ficha y reseñas?"
]


def run_chatbot_evaluation(listing_id, questions=TEST_QUESTIONS, provider=LLM_PROVIDER, top_k=TOP_K):
    """
    Ejecuta preguntas de prueba del módulo LLM/RAG.

    La evaluación automática se realizará en la siguiente celda usando:
    - pregunta;
    - respuesta;
    - evidencias recuperadas;
    - texto completo de evidencia.
    """
    rows = []

    for q in questions:
        try:
            result = answer_question(listing_id, q, top_k=top_k, provider=provider)
            retrieved = result["retrieved_docs"]

            sources = "; ".join([
                f"{d['source']}:{d['source_id']}"
                for d in retrieved[:5]
            ])

            evidencia_texto = "\n\n".join([
                f"[{d['source']} | {d['source_id']}]\n{d['text']}"
                for d in retrieved[:5]
            ])

            rows.append({
                "listing_id": str(listing_id),
                "pregunta": q,
                "respuesta": result["answer"],
                "n_evidencias": len(retrieved),
                "fuentes_recuperadas": sources,
                "evidencia_texto": evidencia_texto
            })

        except Exception as e:
            rows.append({
                "listing_id": str(listing_id),
                "pregunta": q,
                "respuesta": f"ERROR: {repr(e)}",
                "n_evidencias": 0,
                "fuentes_recuperadas": "",
                "evidencia_texto": ""
            })

    return pd.DataFrame(rows)


eval_df = run_chatbot_evaluation(LISTING_ID_DEMO, provider=LLM_PROVIDER)

# Vista resumida para no saturar la pantalla con el texto completo de evidencia.
eval_df_resumen = eval_df.drop(columns=["evidencia_texto"])

display(eval_df_resumen)

,listing_id,pregunta,respuesta,n_evidencias,fuentes_recuperadas
0,708973846265169428,¿El departamento es bueno para trabajar remoto?,La información proporcionada no menciona específicamente si el departamento es adecuado para trabajar remoto. Sin em...,5,reseña:review_1554_chunk_1; reseña:review_1576_chunk_1; ficha_principal:ficha_2; reseña:review_1589_chunk_1; reseña:...
1,708973846265169428,¿Hay quejas frecuentes en las reseñas?,No se identifican quejas frecuentes en las reseñas. Solo hay una crítica mencionada en la reseña del ID review_1578_...,5,reseña:review_1578_chunk_1; ficha_principal:ficha_1; ficha_principal:ficha_2; reseña:review_1590_chunk_1; reseña:rev...
2,708973846265169428,¿Qué aspectos positivos destacan los huéspedes?,"Según las reseñas disponibles, los huéspedes destacan la tranquilidad del lugar (reseña 1), la buena ubicación (rese...",5,ficha_principal:ficha_1; reseña:review_1576_chunk_1; reseña:review_1570_chunk_1; reseña:review_1589_chunk_1; reseña:...
3,708973846265169428,¿El precio parece razonable para lo que ofrece?,"Según la ficha del listado (Ficha_principal | id=ficha_2), el precio por noche es de 91 soles. Sin embargo, no hay s...",5,reseña:review_1563_chunk_1; reseña:review_1576_chunk_1; reseña:review_1579_chunk_1; reseña:review_1569_chunk_1; fich...
4,708973846265169428,¿Qué debería mejorar el host?,No se identifican mejoras específicas con la evidencia disponible. Las reseñas mencionan que el departamento es nuev...,5,reseña:review_1578_chunk_1; ficha_principal:ficha_1; ficha_principal:ficha_2; reseña:review_1590_chunk_1; reseña:rev...
5,708973846265169428,¿La ubicación parece ser una fortaleza del listado?,"Sí, la ubicación parece ser una de las fortalezas del listado. Según el contexto proporcionado, el apartamento está ...",6,ficha_principal:ficha_1; reseña:review_1576_chunk_1; reseña:review_1569_chunk_1; reseña:review_1571_chunk_1; reseña:...
6,708973846265169428,¿El alojamiento parece adecuado para familias o grupos?,"La respuesta es no. La ficha del listado indica que el alojamiento es apto para 2 huéspedes, lo que sugiere que no e...",5,ficha_principal:ficha_1; reseña:review_1583_chunk_1; reseña:review_1557_chunk_1; reseña:review_1559_chunk_1; reseña:...
7,708973846265169428,¿Conviene administrar este departamento según la ficha y reseñas?,"Sí, conviene administrar este departamento. La ficha del listado menciona que el anfitrión, Diego Alberto, tiene la ...",6,ficha_principal:ficha_1; reseña:review_1585_chunk_1; reseña:review_1559_chunk_1; reseña:review_1567_chunk_1; reseña:...


## Evaluación automática basada en reglas

Esta evaluación evita depender de una calificación manual en cada corrida. Como las respuestas generadas por Ollama pueden variar ligeramente, se evalúan criterios estructurales:

1. Si existe evidencia recuperada.
2. Si la respuesta atiende la intención de la pregunta.
3. Si la respuesta evita inferencias no sustentadas por la evidencia.
4. Si ante ausencia de evidencia explícita declara insuficiencia de información.

La evaluación automática no reemplaza por completo la revisión humana, pero ofrece un criterio reproducible para calcular KPIs del módulo LLM/RAG.

In [23]:
# ============================================================
# Evaluación automática del módulo LLM/RAG
# ============================================================
# Esta celda evalúa la respuesta generada usando reglas sobre:
# - pregunta;
# - respuesta;
# - evidencia recuperada.
#
# No usa seed ni fuerza una respuesta exacta del LLM.
# Además, detecta uso indebido de scores técnicos de recuperación
# como si fueran datos del alojamiento.
# ============================================================

import re
import numpy as np
import pandas as pd

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x

def contains_any(text, keywords):
    text = norm_text(text)
    return any(k.lower() in text for k in keywords)

def detect_technical_score_misuse(text):
    """
    Detecta si el LLM está usando scores técnicos del recuperador
    como si fueran puntajes, calificaciones o datos del alojamiento.
    """
    text = "" if pd.isna(text) else str(text).lower()

    patterns = [
        r"puntaje\s+de\s+0\.\d+",
        r"score\s+de\s+0\.\d+",
        r"score\s*[=:]\s*0\.\d+",
        r"similitud\s+de\s+0\.\d+",
        r"similaridad\s+de\s+0\.\d+",
        r"relevancia\s+de\s+0\.\d+",
        r"con\s+un\s+puntaje\s+de\s+0\.\d+"
    ]

    return any(re.search(p, text) for p in patterns)

def evaluate_row_auto(row):
    pregunta = norm_text(row["pregunta"])
    respuesta = norm_text(row["respuesta"])
    evidencia = norm_text(row.get("evidencia_texto", ""))

    reconoce_insuficiencia = contains_any(
        respuesta,
        [
            "no menciona",
            "no hay evidencia",
            "no se menciona",
            "no está explícitamente",
            "no esta explicitamente",
            "no cuenta con información suficiente",
            "no cuenta con informacion suficiente",
            "no hay información suficiente",
            "no hay informacion suficiente",
            "no se puede determinar",
            "no puedo determinar",
            "no se identifican",
            "no se observan"
        ]
    )

    usa_evidencia = row.get("n_evidencias", 0) >= 1 and len(evidencia) > 30

    evidencia_correcta = "Sí" if usa_evidencia else "No"
    respuesta_suficiente = "Sí"
    sin_alucinacion = "Sí"
    observacion = "Evaluación automática basada en respuesta y evidencia recuperada."

    # Control transversal: uso indebido de scores técnicos
    if detect_technical_score_misuse(respuesta):
        return pd.Series({
            "evaluacion_auto_evidencia_correcta": evidencia_correcta,
            "evaluacion_auto_respuesta_suficiente": "No",
            "evaluacion_auto_sin_alucinacion": "No",
            "observacion_auto": (
                "Se detectó posible uso indebido de un score técnico de recuperación "
                "como si fuera dato del alojamiento."
            )
        })

    # 1. Trabajo remoto
    if "trabajar remoto" in pregunta or "remoto" in pregunta:
        if contains_any(evidencia, ["wifi", "internet", "escritorio", "trabajo remoto", "trabajar", "laptop"]):
            observacion = "La evidencia contiene señales relacionadas con trabajo remoto."
        else:
            respuesta_suficiente = "Sí" if reconoce_insuficiencia else "No"
            sin_alucinacion = "Sí" if reconoce_insuficiencia else "No"
            observacion = "No hay evidencia explícita sobre trabajo remoto; se espera que el modelo declare insuficiencia."

    # 2. Quejas frecuentes
    elif "quejas" in pregunta or "frecuentes" in pregunta:
        negative_terms = [
            "problema", "ruido", "sucio", "sucia", "privacidad", "incómodo", "incomodo",
            "demora", "malo", "mala", "queja", "falló", "fallo", "deficiente",
            "sin embargo", "pero", "mejorar"
        ]

        if contains_any(evidencia, negative_terms):
            # Si hay señales negativas, la respuesta debe mencionarlas o ser prudente.
            if contains_any(respuesta, negative_terms) or reconoce_insuficiencia:
                observacion = "La evidencia contiene posibles señales negativas y la respuesta las trata con prudencia."
            else:
                respuesta_suficiente = "No"
                sin_alucinacion = "No"
                observacion = "La evidencia contiene señales negativas, pero la respuesta no las aborda adecuadamente."
        else:
            respuesta_suficiente = "Sí" if reconoce_insuficiencia else "No"
            sin_alucinacion = "Sí" if reconoce_insuficiencia else "No"
            observacion = "No se observan quejas claras en la evidencia; se espera declaración de insuficiencia."

    # 3. Aspectos positivos
    elif "aspectos positivos" in pregunta or "destacan" in pregunta:
        positive_terms = [
            "ubicación", "ubicacion", "limpio", "limpia", "limpieza", "cómodo", "comodo",
            "comodidad", "tranquilo", "recomendado", "excelente", "amable", "atento",
            "agradable", "ordenado", "organizado", "moderno"
        ]

        respuesta_suficiente = "Sí" if contains_any(respuesta, positive_terms) else "No"
        sin_alucinacion = "Sí" if contains_any(evidencia, positive_terms) else "No"
        observacion = "Se evalúa si menciona fortalezas presentes en reseñas o ficha."

    # 4. Precio
    elif "precio" in pregunta:
        respuesta_suficiente = "Sí" if contains_any(
            respuesta,
            ["precio", "calificación", "calificacion", "rating", "reseñas", "resenas", "razonable", "valor", "relación precio"]
        ) else "No"

        sin_alucinacion = "Sí" if contains_any(
            evidencia,
            ["precio", "calificación", "calificacion", "rating", "reseñas", "resenas", "review", "relación precio"]
        ) else "No"

        observacion = "Se evalúa si usa precio, rating o reseñas para justificar sin interpretar scores técnicos."

    # 5. Mejoras del host
    elif "mejorar" in pregunta or "host" in pregunta:
        negative_terms = [
            "problema", "ruido", "sucio", "sucia", "privacidad", "incómodo", "incomodo",
            "demora", "malo", "mala", "queja", "falló", "fallo", "deficiente",
            "muy cerca", "cerca unos de otros", "sin embargo"
        ]

        if contains_any(evidencia, negative_terms):
            # Si hay evidencia negativa, la respuesta puede sugerir mejora.
            if contains_any(respuesta, negative_terms):
                observacion = "La evidencia contiene posibles oportunidades de mejora y la respuesta las utiliza."
            else:
                respuesta_suficiente = "No"
                sin_alucinacion = "No"
                observacion = "La evidencia contiene señales de mejora, pero la respuesta no las recoge adecuadamente."
        else:
            # Si no hay evidencia negativa, lo correcto es no inventar mejoras.
            if reconoce_insuficiencia or contains_any(
                respuesta,
                ["no se identifican", "no hay evidencia", "no se observan", "no se mencionan"]
            ):
                observacion = "No hay críticas claras; el modelo declara falta de evidencia."
            else:
                respuesta_suficiente = "No"
                sin_alucinacion = "No"
                observacion = "El modelo sugiere mejoras no claramente sustentadas en la evidencia."

    # 6. Ubicación
    elif "ubicación" in pregunta or "ubicacion" in pregunta:
        location_terms = ["ubicación", "ubicacion", "céntrico", "centrico", "zona", "cerca", "barranco", "supermercado", "restaurante"]

        respuesta_suficiente = "Sí" if contains_any(respuesta, location_terms) else "No"
        sin_alucinacion = "Sí" if contains_any(evidencia, location_terms) else "No"
        observacion = "Se evalúa si la respuesta se apoya en ficha o reseñas sobre ubicación."

    # 7. Familias o grupos
    elif "familias" in pregunta or "grupos" in pregunta:
        capacity_terms = [
            "familia", "grupo", "huéspedes", "huespedes", "cama", "habitación",
            "habitacion", "baño", "bano", "2 huéspedes", "2 huespedes"
        ]

        if contains_any(evidencia, capacity_terms):
            observacion = "La evidencia contiene señales de capacidad o distribución."
        else:
            respuesta_suficiente = "Sí" if reconoce_insuficiencia else "No"
            sin_alucinacion = "Sí" if reconoce_insuficiencia else "No"
            observacion = "No hay evidencia explícita sobre familias o grupos."

    # 8. Conveniencia comercial
    elif "conviene administrar" in pregunta:
        business_terms_response = [
            "conviene", "administrar", "recomend", "superanfitrión", "superanfitrion",
            "calificación", "calificacion", "reseñas", "resenas", "rating", "positivo"
        ]

        business_terms_evidence = [
            "superanfitrión", "superanfitrion", "calificación", "calificacion",
            "rating", "reseñas", "resenas", "excelente", "recomendado",
            "limpio", "ubicación", "ubicacion", "cómodo", "comodo"
        ]

        respuesta_suficiente = "Sí" if contains_any(respuesta, business_terms_response) else "No"
        sin_alucinacion = "Sí" if contains_any(evidencia, business_terms_evidence) else "No"
        observacion = "Se evalúa si la recomendación comercial se apoya en ficha y reseñas."

    return pd.Series({
        "evaluacion_auto_evidencia_correcta": evidencia_correcta,
        "evaluacion_auto_respuesta_suficiente": respuesta_suficiente,
        "evaluacion_auto_sin_alucinacion": sin_alucinacion,
        "observacion_auto": observacion
    })

eval_auto = eval_df.copy()
auto_cols = eval_auto.apply(evaluate_row_auto, axis=1)
eval_auto = pd.concat([eval_auto, auto_cols], axis=1)

# Vista resumida para lectura en notebook.
eval_auto_resumen = eval_auto.drop(columns=["evidencia_texto"])

display(eval_auto_resumen)

,listing_id,pregunta,respuesta,n_evidencias,fuentes_recuperadas,evaluacion_auto_evidencia_correcta,evaluacion_auto_respuesta_suficiente,evaluacion_auto_sin_alucinacion,observacion_auto
0,708973846265169428,¿El departamento es bueno para trabajar remoto?,La información proporcionada no menciona específicamente si el departamento es adecuado para trabajar remoto. Sin em...,5,reseña:review_1554_chunk_1; reseña:review_1576_chunk_1; ficha_principal:ficha_2; reseña:review_1589_chunk_1; reseña:...,Sí,Sí,Sí,No hay evidencia explícita sobre trabajo remoto; se espera que el modelo declare insuficiencia.
1,708973846265169428,¿Hay quejas frecuentes en las reseñas?,No se identifican quejas frecuentes en las reseñas. Solo hay una crítica mencionada en la reseña del ID review_1578_...,5,reseña:review_1578_chunk_1; ficha_principal:ficha_1; ficha_principal:ficha_2; reseña:review_1590_chunk_1; reseña:rev...,Sí,Sí,Sí,La evidencia contiene posibles señales negativas y la respuesta las trata con prudencia.
2,708973846265169428,¿Qué aspectos positivos destacan los huéspedes?,"Según las reseñas disponibles, los huéspedes destacan la tranquilidad del lugar (reseña 1), la buena ubicación (rese...",5,ficha_principal:ficha_1; reseña:review_1576_chunk_1; reseña:review_1570_chunk_1; reseña:review_1589_chunk_1; reseña:...,Sí,Sí,Sí,Se evalúa si menciona fortalezas presentes en reseñas o ficha.
3,708973846265169428,¿El precio parece razonable para lo que ofrece?,"Según la ficha del listado (Ficha_principal | id=ficha_2), el precio por noche es de 91 soles. Sin embargo, no hay s...",5,reseña:review_1563_chunk_1; reseña:review_1576_chunk_1; reseña:review_1579_chunk_1; reseña:review_1569_chunk_1; fich...,Sí,Sí,Sí,"Se evalúa si usa precio, rating o reseñas para justificar sin interpretar scores técnicos."
4,708973846265169428,¿Qué debería mejorar el host?,No se identifican mejoras específicas con la evidencia disponible. Las reseñas mencionan que el departamento es nuev...,5,reseña:review_1578_chunk_1; ficha_principal:ficha_1; ficha_principal:ficha_2; reseña:review_1590_chunk_1; reseña:rev...,Sí,Sí,Sí,La evidencia contiene posibles oportunidades de mejora y la respuesta las utiliza.
5,708973846265169428,¿La ubicación parece ser una fortaleza del listado?,"Sí, la ubicación parece ser una de las fortalezas del listado. Según el contexto proporcionado, el apartamento está ...",6,ficha_principal:ficha_1; reseña:review_1576_chunk_1; reseña:review_1569_chunk_1; reseña:review_1571_chunk_1; reseña:...,Sí,Sí,Sí,Se evalúa si la respuesta se apoya en ficha o reseñas sobre ubicación.
6,708973846265169428,¿El alojamiento parece adecuado para familias o grupos?,"La respuesta es no. La ficha del listado indica que el alojamiento es apto para 2 huéspedes, lo que sugiere que no e...",5,ficha_principal:ficha_1; reseña:review_1583_chunk_1; reseña:review_1557_chunk_1; reseña:review_1559_chunk_1; reseña:...,Sí,Sí,Sí,La evidencia contiene señales de capacidad o distribución.
7,708973846265169428,¿Conviene administrar este departamento según la ficha y reseñas?,"Sí, conviene administrar este departamento. La ficha del listado menciona que el anfitrión, Diego Alberto, tiene la ...",6,ficha_principal:ficha_1; reseña:review_1585_chunk_1; reseña:review_1559_chunk_1; reseña:review_1567_chunk_1; reseña:...,Sí,Sí,Sí,Se evalúa si la recomendación comercial se apoya en ficha y reseñas.


In [25]:
# ============================================================
# KPIs automáticos del módulo LLM/RAG
# ============================================================

def rate_yes(df, col):
    return round((df[col] == "Sí").mean(), 3)

kpi_auto_llm = pd.DataFrame([
    {
        "KPI": "Tasa automática de evidencia correcta",
        "Definición": "Porcentaje de respuestas con evidencia recuperada disponible.",
        "Valor": rate_yes(eval_auto, "evaluacion_auto_evidencia_correcta")
    },
    {
        "KPI": "Tasa automática de respuesta suficiente",
        "Definición": "Porcentaje de respuestas que atienden la intención de la pregunta.",
        "Valor": rate_yes(eval_auto, "evaluacion_auto_respuesta_suficiente")
    },
    {
        "KPI": "Tasa automática de no alucinación",
        "Definición": "Porcentaje de respuestas sin inferencias no sustentadas según reglas.",
        "Valor": rate_yes(eval_auto, "evaluacion_auto_sin_alucinacion")
    }
])

display(kpi_auto_llm)

,KPI,Definición,Valor
0,Tasa automática de evidencia correcta,Porcentaje de respuestas con evidencia recuperada disponible.,1.0
1,Tasa automática de respuesta suficiente,Porcentaje de respuestas que atienden la intención de la pregunta.,1.0
2,Tasa automática de no alucinación,Porcentaje de respuestas sin inferencias no sustentadas según reglas.,1.0


## 15. Guardar resultados de evaluación

In [26]:
# Exporta la evaluación para incluirla como evidencia en el informe.
OUTPUT_EVAL_PATH = f"evaluacion_chatbot_listing_{LISTING_ID_DEMO}.xlsx"

eval_df.to_excel(OUTPUT_EVAL_PATH, index=False)

print("Evaluación exportada a:", OUTPUT_EVAL_PATH)

Evaluación exportada a: evaluacion_chatbot_listing_708973846265169428.xlsx


## 16. Función puente para integración multimodal

Esta función resume la salida textual del LLM para que luego pueda integrarse con:

- MLP: predicción de rating esperado.
- CNN: score/calidad visual.
- Fusión tardía: score final y recomendación comercial.

In [27]:
def chatbot_summary_for_multimodal(listing_id, provider=LLM_PROVIDER):
    """
    Genera un resumen textual orientado a decisión comercial.
    Luego puede integrarse con MLP y CNN.
    """
    decision_question = """
    Resume la evidencia textual del listado para una decisión comercial.
    Indica:
    1. fortalezas según ficha y reseñas;
    2. debilidades o riesgos;
    3. señales sobre limpieza, ubicación, ruido, comodidad, host y precio;
    4. recomendación textual preliminar: aceptar, aceptar con mejoras o revisar antes de aceptar.
    """

    result = answer_question(
        listing_id=listing_id,
        question=decision_question,
        top_k=8,
        provider=provider
    )

    return result


summary_result = chatbot_summary_for_multimodal(LISTING_ID_DEMO, provider=LLM_PROVIDER)
display_chatbot_result(summary_result)

### Pregunta

    Resume la evidencia textual del listado para una decisión comercial.
    Indica:
    1. fortalezas según ficha y reseñas;
    2. debilidades o riesgos;
    3. señales sobre limpieza, ubicación, ruido, comodidad, host y precio;
    4. recomendación textual preliminar: aceptar, aceptar con mejoras o revisar antes de aceptar.
    

### Respuesta
**Resumen de la evidencia textual del listado**

**Fortalezas:**

* El lugar cumple con la descripción y las fotos (Fuente 1)
* Buena relación precio/calidad (Fuente 1)
* Lugar tranquilo con muy buena ubicación (Fuente 2)
* Excelente ambiente, servicio y buena ubicación (Fuente 3)
* El departamento es nuevo, céntrico y cuenta con servicios adicionales como streaming y música (Fuente 5)
* El host siempre estuvo atento y dispuesto a ayudar (Fuente 4)

**Debilidades o riesgos:**

* Los departamentos están muy cerca unos de otros, lo que puede afectar la privacidad (Fuente 4)
* No se identifican mejoras específicas en las reseñas

**Señales sobre limpieza, ubicación, ruido, comodidad, host y precio:**

* El departamento es moderno y cálido, con infraestructura hogareña (Fuente 7)
* La ubicación es céntrica, cerca de supermercados y mercados tradicionales (Fuente 7)
* No se menciona ruido en las reseñas
* El host tiene una categoría de Superanfitrión y la política de cancelación es flexible (Fuente 5)
* El precio promedio por noche es de 91 soles

**Recomendación textual preliminar:**

Acepta el listado. La evidencia textual sugiere que el lugar cumple con las expectativas de los huéspedes, tiene una buena ubicación y cuenta con servicios adicionales. Aunque se identificó un riesgo en términos de privacidad, no se mencionan críticas significativas en las reseñas. El host también parece ser atento y dispuesto a ayudar.

### Evidencia recuperada

**Fuente 1:** `reseña` — `review_1563_chunk_1` — score: `0.197` — retriever: `tfidf_fallback_por_error_embeddings`

> El lugar cumple con la descripción, es tal cual las fotos, si lo recomiendo. Buena relación precio/calidad.

**Fuente 2:** `reseña` — `review_1576_chunk_1` — score: `0.110` — retriever: `tfidf_fallback_por_error_embeddings`

> 100% recomendado, lugar tranquilo con muy buena ubicación.

...

**Fuente 4:** `reseña` — `review_1578_chunk_1` — score: `0.064` — retriever: `tfidf_fallback_por_error_embeddings`

> El departamento es nuevo, céntrico, sin embargo los departamentos están muy cerca unos con otros. Los vecinos podían ver todo lo que hacíamos desde el balcón del costado. No se tenía mucha privacidad. Sin embargo el Host siempre estuvo muy atento.

**Fuente 5:** `ficha_principal` — `ficha_2` — score: `0.061` — retriever: `tfidf_fallback_por_error_embeddings`

> Streaming (Netflix), Música (Spotify), Soporte tecnológico (Alexa) Adicionales: Cervezas Premium (En la nevera), Cochera (sujeta a disponibilidad), Zona de parrilla( sujeta a disponibilidad) Acceso de los huéspedes Entrada independiente al edificio y ayuda cordial en recepción para una cómoda estadía. ¿Es superhost?: SI ¿Identidad verificada del host?: SI ¿Host tiene foto de perfil?: SI Tiempo como host en años: 0.75 Ubicación exacta: SI Tipo de cama: 1 Número de cuartos o habitaciones: 1 Número de servicios o amenidades: 28 Precio por noche: 91 Promedio de reviews o calificaión: 4.88 Política de cancelación: 1 Verificar teléfono de huésped: SI Número de reviews o reseñas: 42

**Fuente 6:** `reseña` — `review_1571_chunk_1` — score: `0.060` — retriever: `tfidf_fallback_por_error_embeddings`

> ¡Increíble lugar! Tiene todo lo necesario y está en una ubicación agradable.

**Fuente 7:** `ficha_principal` — `ficha_1` — score: `0.049` — retriever: `tfidf_fallback_por_error_embeddings`

> FICHA DEL LISTADO ID Airbnb: 708973846265169428 Distrito: Barranco Título de cabecera: Cálido, céntrico, privado y moderno departamento Título de la propiedad: Apartamento con servicios incluidos entero en Barranco, Perú Resumen de la propiedad: 2 huéspedes - 1 habitación - 1 cama - 1 baño Reconocimiento de Airbnb: Llegada autónoma El personal del edificio te ayudará a hacer el check-in. Diego Alberto tiene la categoría de Superanfitrión Los Superanfitriones son anfitriones experimentados, con evaluaciones excelentes. Las mascotas son bienvenidas Lleva a tus mascotas al alojamiento. Acerca de este espacio: Moderno (1 año de estreno), céntrico (3 cuadras de supermercados y 2 cuadras de un mercado tradicional) y cálido(Infraestructura hogareña) Cuenta con: Juegos de mesa, Streaming (Netflix), Música (Spotify), Soporte tecnológico (Alexa) Adicionales: Cervezas Premium (En la nevera), Cocher

**Fuente 8:** `reseña` — `review_1589_chunk_1` — score: `0.044` — retriever: `tfidf_fallback_por_error_embeddings`

> El departamente muy bonito, limpio y ordenado. Lo recomiendo.

## Fortalezas, debilidades, oportunidades de mejora y recomendación de uso

| Elemento | Análisis del módulo LLM/RAG |
|---|---|
| Fortalezas | Responde en lenguaje natural, usa evidencia del listing seleccionado, muestra fuentes recuperadas y evita mezclar reseñas de distintos alojamientos. |
| Debilidades | Las respuestas pueden variar ligeramente entre ejecuciones, como es normal en un LLM. Además, depende de la calidad de reseñas disponibles y de que Ollama esté activo. |
| Oportunidades de mejora | Incorporar embeddings persistentes, cache de evidencias, evaluación automática más sofisticada y un modo futuro de comparación entre alojamientos. |
| Recomendación de uso futuro | Usar el chatbot como capa explicativa de la decisión multimodal, integrando los resultados de MLP y CNN para justificar recomendaciones comerciales. |

La evaluación automática incorporada permite medir el desempeño del módulo LLM sin depender exclusivamente de una calificación manual corrida por corrida.


## 17. Próximos pasos

1. Reemplazar el Excel por la versión final actualizada del grupo.
2. Validar que las hojas y columnas se detecten correctamente.
3. Elegir 3–5 listings para demo.
4. Configurar proveedor LLM si se requiere respuesta generativa.
5. Completar la evaluación manual de 8–10 preguntas.
6. Integrar con:
   - predicción MLP del rating esperado;
   - clasificación CNN de calidad visual;
   - score final multimodal;
   - recomendación comercial.
7. Migrar a Streamlit cuando el MVP ya funcione.

Sugerencia para la presentación:
- Mostrar que el chatbot no responde desde conocimiento general.
- Mostrar evidencia recuperada.
- Mostrar que, si no hay reseñas o evidencia suficiente, el sistema lo declara.

## 18. Cambios incorporados en v7 respecto al aplicativo

Esta versión del notebook refleja: detección de intención actual, selección de evidencia por intención, separación entre evidencia usada y reseñas recuperadas no usadas, ocultamiento de candidatas si no hay reseñas usadas, reglas condicionales de mejoras y quejas, y scores técnicos fuera del prompt.
